# Notebook 04 — IT Feedback and Multi-Glance Loop (D1–D4)

**Ablations:** D1 (single glance) · D2 (multi-glance, soft/differentiable) ·
D3 (multi-glance, REINFORCE hard attention) · D4 (entropy halting + counterfactual)  
**New module:** `src/it_feedback.py` — `FoveatedModel`, `FixationGrid`, halters  
**Outputs:** `results/04_metrics.json` · `results/04_efficiency_curve.png` ·
`results/04_glance_trajectory.png`

---

## Purpose

The IT-feedback loop (component C5) enables **active, task-driven vision**: instead of
processing the full image uniformly, the model sequentially fixates high-information
locations, accumulates evidence, and halts when confidence is sufficient.

Three variants:
- **D1 (single glance):** standard feedforward, foveation at image centre only.
- **D2 (soft/differentiable):** multi-glance over a fixed grid, soft confidence
  weighting — fully differentiable, trained end-to-end.
- **D3 (REINFORCE):** fixation locations sampled from a learned policy; reward = correct
  classification. Gradient estimated via REINFORCE.
- **D4 (entropy + counterfactual):** uncertainty-driven halting +
  counterfactual loss that rewards informative fixations.

## Mathematics

**Confidence and entropy:**

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \quad
c = \max_i p_i, \quad
H(p) = -\sum_i p_i \log p_i
$$

**Halting condition (D2):** stop after glance $t$ if $c \ge \tau$ or $t = T_{\max}$.

**Soft evidence accumulation (D2):** differentiable weighted sum of glance logits:

$$
\bar z = \frac{1}{T}\sum_{t=1}^T z_t, \quad
\mathcal L = \mathrm{CE}(\bar z,\, y)
$$

**REINFORCE gradient (D3):**

$$
J = \mathbb{E}_{\pi_\phi}[R], \quad
\nabla_\phi J = \mathbb{E}\!\left[(R - b)\,\nabla_\phi \log \pi_\phi(l \mid s)\right]
$$

where $b$ is a baseline (running mean reward) to reduce variance.

**Counterfactual reward (D4):**

$$
r_t^{\mathrm{CF}} = p_{y}^{(t)} - p_{y}^{(t-1)} \quad \text{(correct-class prob gain per glance)}
$$


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json, importlib
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else:
        PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, datasets, models, foveation as fov_mod
from src.foveation import apply_rblur, eccentricity_map, FoveatedTransform, build_foveated_transform, SNRProfile

CFG = {
    "seed": 42, "smoke_test": True,
    "dataset": "cifar10", "image_size": 32, "n_classes": 10,
    "ppd": 4.0, "fixation_yx": (0.5, 0.5),
    "fovea_deg": 1.0, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 1.5,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 8,
    # Multi-glance
    "max_glances": 4, "halt_threshold": 0.75,
    "fixation_grid_rows": 2, "fixation_grid_cols": 2,
    "margin": 0.25,
    # Training
    "n_epochs_smoke": 3, "n_batches_smoke": 20,
    "batch_size": 32, "lr": 1e-3,
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":   os.path.join(PROJECT_ROOT, "checkpoints"),
    "data_dir":   os.path.join(PROJECT_ROOT, "data"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Implementation: FixationGrid and Multi-Glance Components

### Fixation grid

The simplest fixation policy (for D2) uses a regular $n_r \times n_c$ grid:

$$
l_{ij} = \left(\mathrm{margin} + \frac{i}{n_r - 1}(1 - 2\,\mathrm{margin}),\;
                \mathrm{margin} + \frac{j}{n_c - 1}(1 - 2\,\mathrm{margin})\right)
$$

Always includes the image centre $(0.5, 0.5)$ as the first fixation (D1 behaviour).

### Soft evidence accumulation

At each glance $t$ with fixation $l_t$:
1. Foveate image at $l_t$: $x_t = \mathrm{RBlur}(x,\, l_t)$
2. Classify: $z_t = f_\theta(x_t) \in \mathbb{R}^K$
3. Accumulate: $\bar z_t = \frac{1}{t}\sum_{s=1}^{t} z_s$
4. Halt if $\max_i \mathrm{softmax}(\bar z_t)_i \ge \tau$

Loss: $\mathcal{L} = \mathrm{CE}(\bar z_{T_{\mathrm{stop}}},\, y)$


In [ ]:
# ── Fixation grid and multi-glance foveated model ────────────────────────

class FixationGrid:
    """Regular fixation grid always including the centre as first fixation."""

    def __init__(self, n_rows=2, n_cols=2, margin=0.25):
        self.n_rows  = n_rows
        self.n_cols  = n_cols
        self.margin  = margin

    def get_fixations(self):
        """Return list of (fy, fx) fixation coords in [0,1]^2, centre first."""
        m = self.margin
        ys = [m + i / max(self.n_rows - 1, 1) * (1 - 2 * m)
              for i in range(self.n_rows)]
        xs = [m + j / max(self.n_cols - 1, 1) * (1 - 2 * m)
              for j in range(self.n_cols)]
        grid = [(y, x) for y in ys for x in xs]
        # Ensure (0.5, 0.5) is first (centre = D1 baseline)
        centre = (0.5, 0.5)
        grid = sorted(grid, key=lambda p: abs(p[0]-0.5)+abs(p[1]-0.5))
        return grid[:self.n_rows * self.n_cols]


class MultiGlanceFoveatedModel(nn.Module):
    """Differentiable multi-glance foveated classifier (Ablation D2).

    At each glance t:
      1. Foveate image at fixation l_t using R-Blur.
      2. Classify with a shared backbone f_theta.
      3. Accumulate mean logits: z_bar_t = mean(z_1...z_t).
      4. Halt if confidence max softmax(z_bar_t) >= threshold (eval only).

    Loss = CE(z_bar_T, y) -- fully differentiable w.r.t. backbone params.
    """

    def __init__(self, backbone_fn, fixations, halt_threshold=0.75,
                 sigma0=0.5, slope=1.5, ppd=4.0):
        super().__init__()
        self.backbone = backbone_fn          # nn.Module, raw [0,1] input
        self.fixations = fixations           # list of (fy, fx)
        self.halt_threshold = halt_threshold
        self.sigma0 = sigma0
        self.slope  = slope
        self.ppd    = ppd

    def foveate(self, x, fixation_yx):
        """Apply R-Blur at the given fixation to each image in the batch."""
        out = []
        for i in range(x.size(0)):
            out.append(apply_rblur(x[i], fixation_yx, self.sigma0,
                                   self.slope, self.ppd))
        return torch.stack(out)

    def forward(self, x_raw, return_glance_count=False):
        """
        x_raw: [B, C, H, W] raw [0,1] float tensor.
        Returns logits [B, K] and optionally glance_count [B].
        """
        B = x_raw.size(0)
        logit_sum = None
        glance_count = torch.ones(B, dtype=torch.long)
        halted = torch.zeros(B, dtype=torch.bool)

        for t, fix in enumerate(self.fixations):
            fov_x = self.foveate(x_raw, fix)              # [B, C, H, W]
            logits_t = self.backbone(fov_x)                # [B, K]

            if logit_sum is None:
                logit_sum = logits_t
            else:
                logit_sum = logit_sum + logits_t

            avg_logits = logit_sum / (t + 1)

            # Halting (eval mode only — training runs all glances)
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(avg_logits, dim=1).max(dim=1).values  # [B]
                newly_halted = (conf >= self.halt_threshold) & (~halted)
                glance_count += (~halted & ~newly_halted).long()
                halted |= newly_halted
                if halted.all():
                    break

        final_logits = logit_sum / len(self.fixations)
        if return_glance_count:
            return final_logits, glance_count
        return final_logits


# ── Quick test ───────────────────────────────────────────────────────────────
common.set_seed(CFG["seed"])
grid = FixationGrid(CFG["fixation_grid_rows"], CFG["fixation_grid_cols"], CFG["margin"])
fixations = grid.get_fixations()
print(f"Fixation grid ({CFG['fixation_grid_rows']}x{CFG['fixation_grid_cols']}): {fixations}")

S = CFG["image_size"]
dummy = torch.rand(2, 3, S, S)

# Verify foveation at different fixation points produces different images
for fix in fixations[:2]:
    fov_img = apply_rblur(dummy[0], fix, CFG["rblur_sigma0"], CFG["rblur_slope"], CFG["ppd"])
    print(f"  fix={fix}: foveated image norm = {fov_img.norm():.3f}")

print("✓ FixationGrid and foveation OK")


## REINFORCE Hard Attention (D3 — overview)

D3 uses a **learned fixation policy** $\pi_\phi(l \mid s)$ that samples the next
fixation location from a categorical distribution over grid cells:

$$
l_t \sim \pi_\phi(\cdot \mid s_t), \quad
s_t = f_{\mathrm{enc}}(\bar z_{t-1})
$$

The gradient estimator (REINFORCE with baseline):

$$
\nabla_\phi J = \frac{1}{N}\sum_{n=1}^N \sum_{t=1}^{T_n}
\bigl(R_n - b\bigr)\,\nabla_\phi \log \pi_\phi(l_t^{(n)} \mid s_t^{(n)})
$$

where $R_n \in \{0,1\}$ is the classification reward for episode $n$, and
$b = \mathbb{E}[R]$ is a running-mean baseline.

> **Smoke-test scope:** D3 training requires many episodes to converge and is
> excluded from the smoke-test loop. The class below shows the mechanism.


In [ ]:
# ── REINFORCE policy (D3) — mechanism demonstration ──────────────────────
# Full training of D3 requires ~100 epochs. Smoke test only shows one episode.

class FixationPolicy(nn.Module):
    """Lightweight policy network mapping state -> fixation distribution.

    state: accumulated mean logits z_bar (K-dim)
    output: categorical distribution over N_fix fixation locations
    """
    def __init__(self, logit_dim, n_fixations):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(logit_dim, 32), nn.ReLU(),
            nn.Linear(32, n_fixations),
        )
        self.baseline = 0.0    # running-mean reward baseline

    def forward(self, state):
        return F.softmax(self.net(state), dim=-1)   # [B, N_fix]


def reinforce_step(backbone, policy, fixations, x_raw, y,
                    normalise_fn, device_):
    """One REINFORCE episode: sample fixations, compute reward + policy gradient."""
    B = x_raw.size(0)
    n_fix = len(fixations)

    # Initialise with centre fixation (D1-like first glance)
    fov0 = torch.stack([apply_rblur(x_raw[i], fixations[0], 0.5, 1.5, 4.0)
                         for i in range(B)])
    x_norm0 = normalise_fn(fov0)
    with torch.no_grad():
        z_bar = backbone(x_norm0)

    # Sample second fixation from policy
    state = z_bar.detach()
    probs = policy(state)                          # [B, n_fix]
    dist  = torch.distributions.Categorical(probs)
    idx   = dist.sample()                          # [B]
    log_p = dist.log_prob(idx)                     # [B]

    # Apply sampled fixation and classify
    fov_list = []
    for i in range(B):
        fix = fixations[idx[i].item()]
        fov_list.append(apply_rblur(x_raw[i], fix, 0.5, 1.5, 4.0))
    fov_batch = torch.stack(fov_list)
    x_norm1 = normalise_fn(fov_batch)
    with torch.no_grad():
        z2 = backbone(x_norm1)
    logits = (z_bar + z2) / 2
    pred   = logits.argmax(dim=1)
    reward = (pred == y).float()                   # [B]  in {0, 1}

    # REINFORCE gradient
    baseline = policy.baseline
    loss_policy = -((reward - baseline) * log_p).mean()

    # Update baseline (running mean)
    policy.baseline = 0.9 * baseline + 0.1 * reward.mean().item()

    return loss_policy, reward.mean().item()


# Quick smoke test of one REINFORCE episode
S = CFG["image_size"]
dummy_x = torch.rand(4, 3, S, S)
dummy_y = torch.randint(0, CFG["n_classes"], (4,))

def _raw_backbone(x):
    """Dummy backbone: global average pool -> linear (for smoke test only)."""
    return nn.Linear(3, CFG["n_classes"])(x.mean(dim=[2, 3]))

def _norm(x): return x   # identity normalisation for smoke test

tiny_backbone = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                               nn.Linear(3, CFG["n_classes"]))
policy_net = FixationPolicy(logit_dim=CFG["n_classes"], n_fixations=len(fixations))
policy_opt = torch.optim.Adam(policy_net.parameters(), lr=1e-3)

loss_p, reward_val = reinforce_step(
    tiny_backbone, policy_net, fixations, dummy_x, dummy_y,
    normalise_fn=lambda x: x, device_=device
)
print(f"REINFORCE episode: policy_loss={loss_p.item():.4f}  reward={reward_val:.3f}")
print("✓ REINFORCE mechanism verified (smoke test, random model)")


In [ ]:
# ── Synthetic CIFAR-10-like dataset + tiny backbone ──────────────────────

class SyntheticCIFAR(torch.utils.data.Dataset):
    def __init__(self, n=512, H=32, W=32, n_classes=10, seed=42):
        rng = np.random.RandomState(seed)
        self.images = torch.rand(n, 3, H, W)
        self.labels = torch.from_numpy(rng.randint(0, n_classes, n)).long()
        mu  = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        self.images = (self.images - mu) / std   # normalised
    def __len__(self): return len(self.images)
    def __getitem__(self, i): return self.images[i], self.labels[i]


class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
            nn.Flatten(), nn.Linear(64*4*4, n_classes),
        )
    def forward(self, x): return self.net(x)


INV_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
INV_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def denorm(x): return (x * INV_STD + INV_MEAN).clamp(0.0, 1.0)
def renorm(x): return (x - INV_MEAN) / INV_STD

def get_loaders():
    tr = SyntheticCIFAR(n=512, seed=42)
    va = SyntheticCIFAR(n=128, seed=99)
    return (torch.utils.data.DataLoader(tr, batch_size=CFG["batch_size"], shuffle=True),
            torch.utils.data.DataLoader(va, batch_size=CFG["batch_size"], shuffle=False))

train_ld, val_ld = get_loaders()
print(f"Dataloaders ready: {len(train_ld.dataset)} train, {len(val_ld.dataset)} val")


In [ ]:
# ── D1 (single glance) vs D2 (multi-glance) training ─────────────────────

def train_epoch(model, loader, opt, device_, n_batches=None,
                multi_glance=False):
    model.train()
    mu  = torch.tensor([0.485, 0.456, 0.406], device=device_).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device_).view(1,3,1,1)
    correct = total = tot_loss = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches and bi >= n_batches: break
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw  = (xb * std + mu).clamp(0.0, 1.0)   # undo normalisation -> [0,1]

        if multi_glance:
            logits = model(x_raw)
        else:
            # D1: centre fixation only, re-normalise before backbone
            fov_x  = torch.stack([apply_rblur(x_raw[i], (0.5, 0.5),
                                              CFG["rblur_sigma0"], CFG["rblur_slope"],
                                              CFG["ppd"])
                                   for i in range(x_raw.size(0))])
            fov_norm = (fov_x - mu) / std
            logits = model(fov_norm)

        loss = F.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return tot_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def eval_epoch(model, loader, device_, multi_glance=False):
    model.eval()
    mu  = torch.tensor([0.485, 0.456, 0.406], device=device_).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device_).view(1,3,1,1)
    correct = total = total_glances = 0
    for xb, yb in loader:
        xb, yb = xb.to(device_), yb.to(device_)
        x_raw = (xb * std + mu).clamp(0.0, 1.0)
        if multi_glance:
            logits, gc = model(x_raw, return_glance_count=True)
            total_glances += gc.sum().item()
        else:
            fov_x   = torch.stack([apply_rblur(x_raw[i], (0.5, 0.5),
                                               CFG["rblur_sigma0"], CFG["rblur_slope"],
                                               CFG["ppd"])
                                    for i in range(x_raw.size(0))])
            fov_norm = (fov_x - mu) / std
            logits   = model(fov_norm)
            total_glances += xb.size(0)   # always 1 glance
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    avg_glances = total_glances / max(total, 1)
    return correct / max(total, 1), avg_glances


# ── Train D1 ──────────────────────────────────────────────────────────────
common.set_seed(CFG["seed"])
print("=== D1: Single-glance foveated classifier ===")
backbone_d1 = TinyCNN(CFG["n_classes"]).to(device)
opt_d1 = torch.optim.Adam(backbone_d1.parameters(), lr=CFG["lr"])
hist_d1 = []
for epoch in range(CFG["n_epochs_smoke"]):
    tr_loss, tr_acc = train_epoch(backbone_d1, train_ld, opt_d1, device,
                                   n_batches=CFG["n_batches_smoke"])
    val_acc, avg_gc = eval_epoch(backbone_d1, val_ld, device, multi_glance=False)
    hist_d1.append({"epoch": epoch, "val_acc": val_acc, "avg_glances": avg_gc})
    print(f"  E{epoch+1}: loss={tr_loss:.4f} tr={tr_acc:.3f} val={val_acc:.3f} glances={avg_gc:.1f}")

# ── Train D2 ──────────────────────────────────────────────────────────────
common.set_seed(CFG["seed"])
print("\n=== D2: Multi-glance foveated classifier ===")
backbone_d2 = TinyCNN(CFG["n_classes"]).to(device)

# MultiGlanceFoveatedModel wraps backbone to accept raw [0,1] input
# Backbone inside must handle normalised input
class NormBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.register_buffer("mu",  torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))
    def forward(self, x_raw):
        return self.backbone((x_raw - self.mu) / self.std)

mg_model = MultiGlanceFoveatedModel(
    backbone_fn=NormBackbone(backbone_d2),
    fixations=fixations,
    halt_threshold=CFG["halt_threshold"],
    sigma0=CFG["rblur_sigma0"],
    slope=CFG["rblur_slope"],
    ppd=CFG["ppd"],
).to(device)

opt_d2 = torch.optim.Adam(backbone_d2.parameters(), lr=CFG["lr"])
hist_d2 = []
for epoch in range(CFG["n_epochs_smoke"]):
    tr_loss, tr_acc = train_epoch(mg_model, train_ld, opt_d2, device,
                                   n_batches=CFG["n_batches_smoke"], multi_glance=True)
    val_acc, avg_gc = eval_epoch(mg_model, val_ld, device, multi_glance=True)
    hist_d2.append({"epoch": epoch, "val_acc": val_acc, "avg_glances": avg_gc})
    print(f"  E{epoch+1}: loss={tr_loss:.4f} tr={tr_acc:.3f} val={val_acc:.3f} glances={avg_gc:.2f}")


In [ ]:
# ── Visualise: efficiency curve + glance trajectories ─────────────────────

common.set_seed(CFG["seed"])

# Figure 1: accuracy vs avg glances
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
epochs = [h["epoch"] for h in hist_d1]
ax.plot(epochs, [h["val_acc"] for h in hist_d1], "o-", label="D1 single-glance")
ax.plot(epochs, [h["val_acc"] for h in hist_d2], "s-", label="D2 multi-glance")
ax.set_xlabel("Epoch"); ax.set_ylabel("Val accuracy")
ax.set_title("D1 vs D2: accuracy per epoch\n(smoke_test, random model)"); ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
d1_gc = [h["avg_glances"] for h in hist_d1]
d2_gc = [h["avg_glances"] for h in hist_d2]
d1_va = [h["val_acc"]    for h in hist_d1]
d2_va = [h["val_acc"]    for h in hist_d2]
ax.plot(d1_gc, d1_va, "o-", label="D1 (single glance)", markersize=8)
ax.plot(d2_gc, d2_va, "s-", label="D2 (confidence halting)", markersize=8)
ax.set_xlabel("Avg glances per image"); ax.set_ylabel("Val accuracy")
ax.set_title("Efficiency curve: accuracy vs compute\n(glances = FLOPs proxy)")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
eff_path = os.path.join(CFG["result_dir"], "04_efficiency_curve.png")
common.save_figure(fig, eff_path, dpi=130)
plt.close(fig)
print(f"Saved: {eff_path}")

# Figure 2: glance trajectory visualisation on a few images
n_show = 4
fig2, ax2s = plt.subplots(n_show, len(fixations) + 1, figsize=(3*(len(fixations)+1), 3*n_show))
mu_t  = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
std_t = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

test_batch = next(iter(val_ld))
x_norm_batch, y_batch = test_batch
x_raw_batch = (x_norm_batch * std_t + mu_t).clamp(0.0, 1.0)

S = CFG["image_size"]

for row_i in range(min(n_show, x_raw_batch.size(0))):
    img_raw = x_raw_batch[row_i]   # [3, H, W]

    ax2s[row_i][0].imshow(img_raw.permute(1,2,0).numpy().clip(0,1), interpolation="nearest")
    ax2s[row_i][0].set_title("Original", fontsize=8)
    ax2s[row_i][0].axis("off")

    for col_j, fix in enumerate(fixations[:len(fixations)], start=1):
        fov_img = apply_rblur(img_raw, fix, CFG["rblur_sigma0"], CFG["rblur_slope"], CFG["ppd"])
        ax = ax2s[row_i][col_j]
        ax.imshow(fov_img.permute(1,2,0).numpy().clip(0,1), interpolation="nearest")
        fy_px = int(fix[0] * S); fx_px = int(fix[1] * S)
        ax.plot(fx_px, fy_px, "r+", markersize=10, markeredgewidth=2)
        if row_i == 0:
            ax.set_title(f"Glance {col_j}\nfix=({fix[0]:.2f},{fix[1]:.2f})", fontsize=7)
        ax.axis("off")

fig2.suptitle("Glance trajectories: R-Blur at each fixation location", fontsize=10, y=1.01)
plt.tight_layout()
traj_path = os.path.join(CFG["result_dir"], "04_glance_trajectory.png")
common.save_figure(fig2, traj_path, dpi=130)
plt.close(fig2)
print(f"Saved: {traj_path}")


## Write `src/it_feedback.py`

The cell below writes the production module used by NB06 (full model).


In [ ]:
_itfb_content = '"""\nIT feedback and multi-glance loop components.\n\nPublic API\n----------\nFixationGrid           -- regular grid of fixation locations\nMultiGlanceFoveatedModel -- D2: differentiable multi-glance (confidence halting)\nFixationPolicy         -- D3: learned REINFORCE fixation policy\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom src.foveation import apply_rblur\n\n\nclass FixationGrid:\n    """Regular fixation grid, centre fixation always first."""\n    def __init__(self, n_rows=2, n_cols=2, margin=0.25):\n        self.n_rows = n_rows; self.n_cols = n_cols; self.margin = margin\n    def get_fixations(self):\n        m = self.margin\n        ys = [m + i / max(self.n_rows-1, 1)*(1-2*m) for i in range(self.n_rows)]\n        xs = [m + j / max(self.n_cols-1, 1)*(1-2*m) for j in range(self.n_cols)]\n        grid = [(y, x) for y in ys for x in xs]\n        return sorted(grid, key=lambda p: abs(p[0]-0.5)+abs(p[1]-0.5))\n\n\nclass MultiGlanceFoveatedModel(nn.Module):\n    """\n    Differentiable multi-glance foveated classifier (D2).\n    Backbone receives raw [0,1] pixel tensors.\n    """\n    def __init__(self, backbone_fn, fixations, halt_threshold=0.75,\n                 sigma0=0.5, slope=1.5, ppd=4.0):\n        super().__init__()\n        self.backbone        = backbone_fn\n        self.fixations       = fixations\n        self.halt_threshold  = halt_threshold\n        self.sigma0 = sigma0; self.slope = slope; self.ppd = ppd\n    def foveate(self, x, fixation_yx):\n        return torch.stack([apply_rblur(x[i], fixation_yx, self.sigma0,\n                                        self.slope, self.ppd) for i in range(x.size(0))])\n    def forward(self, x_raw, return_glance_count=False):\n        B = x_raw.size(0)\n        logit_sum = None; halted = torch.zeros(B, dtype=torch.bool)\n        glance_count = torch.ones(B, dtype=torch.long)\n        for t, fix in enumerate(self.fixations):\n            fov_x = self.foveate(x_raw, fix)\n            z_t   = self.backbone(fov_x)\n            logit_sum = z_t if logit_sum is None else logit_sum + z_t\n            if not self.training and t < len(self.fixations) - 1:\n                conf = F.softmax(logit_sum / (t+1), dim=1).max(1).values\n                newly = (conf >= self.halt_threshold) & (~halted)\n                glance_count += (~halted & ~newly).long()\n                halted |= newly\n                if halted.all(): break\n        final = logit_sum / len(self.fixations)\n        return (final, glance_count) if return_glance_count else final\n\n\nclass FixationPolicy(nn.Module):\n    """REINFORCE policy: state (logits) -> categorical over fixation locations."""\n    def __init__(self, logit_dim, n_fixations):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(logit_dim, 32), nn.ReLU(), nn.Linear(32, n_fixations))\n        self.baseline = 0.0\n    def forward(self, state):\n        return F.softmax(self.net(state), dim=-1)'
_itfb_path = os.path.join(PROJECT_ROOT, 'src', 'it_feedback.py')
with open(_itfb_path, 'w') as _f: _f.write(_itfb_content)
print(f'Written: {_itfb_path}')

# Verify import
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('it_feedback', _itfb_path)
_mod  = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
assert hasattr(_mod, 'MultiGlanceFoveatedModel'), 'MultiGlanceFoveatedModel missing'
assert hasattr(_mod, 'FixationGrid'),             'FixationGrid missing'
print('src/it_feedback.py: 3 public symbols verified')

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
output = {
    "notebook": "04_it_feedback_multiglance",
    "cfg": {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "D1": {"history": hist_d1, "note": "smoke_test synthetic data"},
    "D2": {"history": hist_d2, "note": "smoke_test synthetic data"},
    "D3": {"note": "TODO: full REINFORCE training (P2, not run in smoke_test)"},
    "D4": {"note": "TODO: entropy halting + counterfactual loss (P3)"},
}
rpath = os.path.join(CFG["result_dir"], "04_metrics.json")
common.save_json(output, rpath)
print(f"Results: {rpath}")
print(f"04_efficiency_curve.png   : {os.path.exists(os.path.join(CFG['result_dir'], '04_efficiency_curve.png'))}")
print(f"04_glance_trajectory.png  : {os.path.exists(os.path.join(CFG['result_dir'], '04_glance_trajectory.png'))}")
print(f"src/it_feedback.py        : {os.path.exists(os.path.join(PROJECT_ROOT,'src','it_feedback.py'))}")
print("\n── Notebook 04 complete ✓")
